## Лабораторная 6. Шпаргалка к защите

### Рассказ на минуту

> Я взял три фотографии комнаты по порядку съёмки. Соседние кадры перекрываются, поэтому на них можно найти общие детали, например дверной проём.

> На каждом кадре SIFT находит ключевые точки и описывает участок вокруг каждой точки набором чисел. По этим описаниям я сопоставляю точки соседних кадров. Критерий Лоу оставляет пару, если лучший кандидат заметно ближе второго. Затем RANSAC подбирает гомографию, матрицу для совмещения кадров, и отбрасывает пары, которые с ней плохо согласуются.

> Первый кадр служит основой. Второй переношу сразу к первому, а третий через второй. Для этого перемножаю две матрицы. По новым координатам углов всех кадров считаю размер холста и переношу на него фотографии. В местах перекрытия смешиваю цвета, чтобы переходы были плавнее.

> В конце по маске покрытия вырезаю самый большой прямоугольник без пустых участков. Получилась панорама размером 2453 × 1200 пикселей.

### Если попросят уточнить

- Лоу проверяет сходство описаний точек. Расстояние до лучшего кандидата должно быть меньше 75% расстояния до второго. RANSAC проверяет геометрию. В коде порог ошибки при переносе точки составляет 3 пикселя.
- Соседние кадры сопоставляются попарно. Чтобы перенести третий в координаты первого, сначала применяю матрицу `3 → 2`, затем `2 → 1`.
- Чёрный пиксель может быть частью фотографии, например рамой беговой дорожки. Маска показывает, попала ли в это место хотя бы одна фотография.
- Справа стеллаж вытянут. При переносе на плоскость первого кадра дальняя часть панорамы растягивается.

Для показа открой сначала [холст с окном обрезки](assets/04_canvas.png), затем [готовую панораму](../panorama.png).

> Подключаю OpenCV для обработки фотографий, NumPy для работы с матрицами и Matplotlib для показа изображений.

In [1]:
%matplotlib inline

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import Rectangle

> Эта функция показывает кадры рядом. Перед выводом перевожу цвета из BGR, который использует OpenCV, в RGB для Matplotlib.

In [2]:
def show_row(images, titles, width=4):
    fig, axes = plt.subplots(
        1, len(images), figsize=(width * len(images), 3.5),
        squeeze=False, layout="constrained",
    )
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    plt.close(fig)
    return fig

> Загружаю три снимка комнаты по порядку съёмки. Уменьшаю каждый до 1200 пикселей по длинной стороне, чтобы быстрее обработать кадры.

> 3. Перевожу кадры в оттенки серого. SIFT находит характерные точки и строит дескрипторы, по которым я ищу те же детали на соседнем снимке. Для показа отмечаю по 200 точек с самым сильным откликом.

> 4. Для каждой точки ищу два ближайших дескриптора в предыдущем кадре. По критерию Лоу оставляю пару, если первый кандидат заметно ближе второго. Затем RANSAC подбирает матрицу совмещения и отсеивает пары, которые с ней плохо согласуются.

> 5. Второй кадр переношу к первому, а третий через второй, перемножая матрицы. По крайним координатам преобразованных углов считаю размер холста. Затем сдвигаю все кадры так, чтобы граница холста начиналась с нуля.

> 6. Переношу на холст фотографии и их маски. В перекрытиях усредняю цвета с весами. Чем дальше пиксель от края своего кадра, тем больше его вклад.

> 7. По маске ищу самый большой прямоугольник, целиком покрытый фотографиями. Для каждой строки считаю, сколько покрытых пикселей подряд идёт вверх в каждом столбце. По этим высотам нахожу самый большой прямоугольник.

> На холсте цветом отмечены границы кадров, а белым пунктиром окно обрезки. Вырезаю его и получаю панораму 2453 на 1200 пикселей. Внутри каждый пиксель покрыт хотя бы одним кадром.

> Сохраняю готовую панораму в `panorama.png` рядом с исходными фотографиями.

In [ ]:
output_path = lab_dir / "panorama.png"
ok, encoded = cv2.imencode(".png", panorama)
if not ok:
    raise RuntimeError("Не удалось закодировать панораму")
output_path.write_bytes(encoded.tobytes())
print(output_path.name)